Cell 1: Load toàn bộ kết quả

In [1]:
# ===== G11 - CELL 1: LOAD FORECAST RESULTS =====

import pandas as pd
import numpy as np

# -----------------------------------------
# Load locked forecasting dataset
# -----------------------------------------
df = pd.read_csv(
    "../output/G06_forecast_dataset.csv"
)

df["Month"] = pd.to_datetime(df["Month"])


# -----------------------------------------
# G06 - Naive / MA
# -----------------------------------------
baseline_predictions = pd.read_csv(
    "../output/G06_baseline_predictions.csv"
)

baseline_metrics = pd.read_csv(
    "../output/G06_baseline_metrics.csv"
)


# -----------------------------------------
# G09 - ETS
# -----------------------------------------
ets_predictions = pd.read_csv(
    "../output/G09_ets_predictions.csv"
)

ets_metrics = pd.read_csv(
    "../output/G09_ets_metrics.csv"
)


# -----------------------------------------
# G10 - ARIMA
# -----------------------------------------
arima_predictions = pd.read_csv(
    "../output/G10_arima_predictions.csv"
)

arima_metrics = pd.read_csv(
    "../output/G10_arima_metrics.csv"
)


# -----------------------------------------
# Convert forecast dates
# -----------------------------------------
baseline_predictions["ForecastMonth"] = pd.to_datetime(
    baseline_predictions["ForecastMonth"]
)

ets_predictions["ForecastMonth"] = pd.to_datetime(
    ets_predictions["ForecastMonth"]
)

arima_predictions["ForecastMonth"] = pd.to_datetime(
    arima_predictions["ForecastMonth"]
)


# -----------------------------------------
# Basic reconciliation
# -----------------------------------------
print("=== G11 INPUT CHECK ===")

print("\nDataset:")
print("Rows:", len(df))
print("Materials:", df["MaterialKey"].nunique())

print("\nG06 Baseline:")
print("Predictions:", len(baseline_predictions))
print("Metrics:", len(baseline_metrics))

print("\nG09 ETS:")
print("Predictions:", len(ets_predictions))
print("Metrics:", len(ets_metrics))

print("\nG10 ARIMA:")
print("Predictions:", len(arima_predictions))
print("Metrics:", len(arima_metrics))

=== G11 INPUT CHECK ===

Dataset:
Rows: 1145
Materials: 35

G06 Baseline:
Predictions: 612
Metrics: 105

G09 ETS:
Predictions: 588
Metrics: 105

G10 ARIMA:
Predictions: 204
Metrics: 35


Cell 2: RECONCILE TEST SET ACROSS MODELS

In [6]:
# ===== G11 - CELL 2: RECONCILE TEST SET ACROSS MODELS =====

# Standard 6-month test set from locked dataset
test_reference = (
    df[df["IsTestPeriod"] == 1]
    [
        [
            "MaterialKey",
            "Month",
            "RequirementQty_PU",
            "UnitCode"
        ]
    ]
    .copy()
)

test_reference = test_reference.rename(
    columns={
        "Month": "ForecastMonth",
        "RequirementQty_PU": "ReferenceActual"
    }
)


def validate_predictions(predictions, name):
    """
    Validate prediction output against the locked G06 test set.
    """

    pred = predictions.copy()

    # Core QA
    duplicate_count = pred.duplicated(
        subset=[
            "MaterialKey",
            "ForecastMonth",
            "Model"
        ]
    ).sum()

    null_actual = pred["ActualQty"].isna().sum()
    null_forecast = pred["ForecastQty"].isna().sum()

    # Compare actual values against locked dataset
    check = pred.merge(
        test_reference,
        on=[
            "MaterialKey",
            "ForecastMonth"
        ],
        how="left"
    )

    missing_reference = (
        check["ReferenceActual"]
        .isna()
        .sum()
    )

    actual_mismatch = (
        ~np.isclose(
            check["ActualQty"],
            check["ReferenceActual"],
            equal_nan=False
        )
    ).sum()

    print(f"\n=== {name} ===")

    print(
        "Prediction rows:",
        len(pred)
    )

    print(
        "Materials:",
        pred["MaterialKey"].nunique()
    )

    print(
        "Models:",
        sorted(pred["Model"].unique())
    )

    print(
        "Duplicate Material-Month-Model:",
        duplicate_count
    )

    print(
        "NULL Actual:",
        null_actual
    )

    print(
        "NULL Forecast:",
        null_forecast
    )

    print(
        "Missing locked-test reference:",
        missing_reference
    )

    print(
        "Actual mismatch vs locked test:",
        actual_mismatch
    )


# -----------------------------------------
# Validate all three model groups
# -----------------------------------------

print("=== G11 TEST-SET RECONCILIATION ===")

validate_predictions(
    baseline_predictions,
    "G06 BASELINE"
)

validate_predictions(
    ets_predictions,
    "G09 ETS"
)

validate_predictions(
    arima_predictions,
    "G10 ARIMA"
)


# ===== CORRECTED TEST COVERAGE =====

print("\n=== TEST COVERAGE ===")

all_materials = (
    df[["MaterialKey"]]
    .drop_duplicates()
    .sort_values("MaterialKey")
)

reference_test_count = (
    test_reference
    .groupby("MaterialKey")
    .size()
    .rename("TestObserved")
    .reset_index()
)

test_coverage = all_materials.merge(
    reference_test_count,
    on="MaterialKey",
    how="left"
)

# Materials with no test rows must be explicitly counted as 0
test_coverage["TestObserved"] = (
    test_coverage["TestObserved"]
    .fillna(0)
    .astype(int)
)

print(
    "Total materials:",
    len(test_coverage)
)

print(
    "Materials with full 6-month test:",
    (test_coverage["TestObserved"] == 6).sum()
)

print(
    "Materials without full 6-month test:",
    (test_coverage["TestObserved"] != 6).sum()
)

print("\nIncomplete test materials:")

display(
    test_coverage[
        test_coverage["TestObserved"] != 6
    ]
)

=== G11 TEST-SET RECONCILIATION ===

=== G06 BASELINE ===
Prediction rows: 612
Materials: 34
Models: ['MA_3', 'MA_6', 'Naive']
Duplicate Material-Month-Model: 0
NULL Actual: 0
NULL Forecast: 0
Missing locked-test reference: 0
Actual mismatch vs locked test: 0

=== G09 ETS ===
Prediction rows: 588
Materials: 34
Models: ['ETS_Level', 'ETS_Seasonal', 'ETS_Trend']
Duplicate Material-Month-Model: 0
NULL Actual: 0
NULL Forecast: 0
Missing locked-test reference: 0
Actual mismatch vs locked test: 0

=== G10 ARIMA ===
Prediction rows: 204
Materials: 34
Models: ['ARIMA']
Duplicate Material-Month-Model: 0
NULL Actual: 0
NULL Forecast: 0
Missing locked-test reference: 0
Actual mismatch vs locked test: 0

=== TEST COVERAGE ===
Total materials: 35
Materials with full 6-month test: 34
Materials without full 6-month test: 1

Incomplete test materials:


,MaterialKey,TestObserved
22,169,0


Cell 3: MODEL SELECTION RULE

In [7]:
# ===== G11 - CELL 3: DEFINE MODEL SELECTION RULE =====

# -----------------------------------------
# Model complexity ranking
# Lower number = simpler model
# -----------------------------------------

MODEL_COMPLEXITY = {
    "Naive": 1,
    "MA_3": 2,
    "MA_6": 3,
    "ETS_Level": 4,
    "ETS_Trend": 5,
    "ETS_Seasonal": 6,
    "ARIMA": 7
}

# Near-tie tolerance:
# Models within 1% of the minimum MAE
# are treated as practically similar.
MAE_TOLERANCE = 0.01


def select_best_model(material_metrics):
    """
    Model-selection rule:

    1. Use successful models only.
    2. MAE is the primary criterion.
    3. Models within 1% of minimum MAE
       are treated as near-ties.
    4. RMSE is the secondary criterion.
    5. If performance remains near-equivalent,
       prefer the simpler model.
    6. Bias is retained as a diagnostic metric,
       not the primary ranking criterion.
    """

    candidates = material_metrics.copy()

    candidates = candidates[
        candidates["Status"] == "SUCCESS"
    ].copy()

    candidates = candidates.dropna(
        subset=["MAE", "RMSE"]
    )

    if len(candidates) == 0:
        return None

    # -----------------------------------------
    # Primary criterion: MAE
    # -----------------------------------------
    min_mae = candidates["MAE"].min()

    mae_limit = min_mae * (1 + MAE_TOLERANCE)

    near_tie = candidates[
        candidates["MAE"] <= mae_limit
    ].copy()

    # -----------------------------------------
    # Secondary criterion: RMSE
    # -----------------------------------------
    min_rmse = near_tie["RMSE"].min()

    rmse_limit = min_rmse * (1 + MAE_TOLERANCE)

    finalists = near_tie[
        near_tie["RMSE"] <= rmse_limit
    ].copy()

    # -----------------------------------------
    # Final tie-break: simpler model
    # -----------------------------------------
    finalists["ComplexityRank"] = (
        finalists["Model"]
        .map(MODEL_COMPLEXITY)
    )

    finalists = finalists.sort_values(
        [
            "ComplexityRank",
            "MAE",
            "RMSE"
        ]
    )

    return finalists.iloc[0]


print("=== MODEL SELECTION RULE ===")
print("Primary criterion: MAE")
print("Near-tie tolerance: 1%")
print("Secondary criterion: RMSE")
print("Final tie-break: simpler model")
print("Bias: diagnostic only")

print("\nModel complexity:")
for model, rank in MODEL_COMPLEXITY.items():
    print(rank, "-", model)

=== MODEL SELECTION RULE ===
Primary criterion: MAE
Near-tie tolerance: 1%
Secondary criterion: RMSE
Final tie-break: simpler model
Bias: diagnostic only

Model complexity:
1 - Naive
2 - MA_3
3 - MA_6
4 - ETS_Level
5 - ETS_Trend
6 - ETS_Seasonal
7 - ARIMA


Cell 4: MERGE ALL MODEL METRICS & SELECT BEST MODEL

In [8]:
# ===== G11 - CELL 4: MERGE METRICS & SELECT BEST MODEL =====

# -----------------------------------------
# Standardize metric columns
# -----------------------------------------

metric_columns = [
    "MaterialKey",
    "Model",
    "MAE",
    "RMSE",
    "Bias",
    "Status"
]

baseline_compare = baseline_metrics[
    metric_columns
].copy()

ets_compare = ets_metrics[
    metric_columns
].copy()

arima_compare = arima_metrics[
    metric_columns
].copy()


# -----------------------------------------
# Combine all model evaluation results
# -----------------------------------------

all_metrics = pd.concat(
    [
        baseline_compare,
        ets_compare,
        arima_compare
    ],
    ignore_index=True
)


print("=== COMBINED MODEL METRICS ===")

print("Rows:", len(all_metrics))

print(
    "Materials:",
    all_metrics["MaterialKey"].nunique()
)

print("\nModels:")
print(
    all_metrics["Model"]
    .value_counts()
    .sort_index()
)

print("\nStatus:")
print(
    all_metrics["Status"]
    .value_counts()
)


# -----------------------------------------
# Select best model for each material
# -----------------------------------------

selection_rows = []

for material_key, material_metrics in all_metrics.groupby(
    "MaterialKey"
):

    winner = select_best_model(
        material_metrics
    )

    if winner is None:

        selection_rows.append({
            "MaterialKey": material_key,
            "SelectedModel": None,
            "MAE": np.nan,
            "RMSE": np.nan,
            "Bias": np.nan,
            "SelectionStatus": "NO_VALIDATED_MODEL"
        })

    else:

        selection_rows.append({
            "MaterialKey": material_key,
            "SelectedModel": winner["Model"],
            "MAE": winner["MAE"],
            "RMSE": winner["RMSE"],
            "Bias": winner["Bias"],
            "SelectionStatus": "SELECTED"
        })


model_selection = pd.DataFrame(
    selection_rows
).sort_values(
    "MaterialKey"
).reset_index(drop=True)


# -----------------------------------------
# Selection summary
# -----------------------------------------

print("\n=== MODEL SELECTION RESULTS ===")

print(
    model_selection["SelectionStatus"]
    .value_counts()
)

print("\nSelected model distribution:")

print(
    model_selection.loc[
        model_selection["SelectionStatus"] == "SELECTED",
        "SelectedModel"
    ]
    .value_counts()
)


print("\nMaterials without validated model:")

display(
    model_selection[
        model_selection["SelectionStatus"]
        != "SELECTED"
    ]
)


print("\n=== SAMPLE SELECTED MODELS ===")

display(
    model_selection.head(10)
)

=== COMBINED MODEL METRICS ===
Rows: 245
Materials: 35

Models:
Model
ARIMA           35
ETS_Level       35
ETS_Seasonal    35
ETS_Trend       35
MA_3            35
MA_6            35
Naive           35
Name: count, dtype: int64

Status:
Status
SUCCESS                                 234
INSUFFICIENT_TEST_DATA_0_OF_6             7
NOT_ELIGIBLE_SEASONAL_TRAIN_6_LT_24       2
NOT_ELIGIBLE_SEASONAL_TRAIN_18_LT_24      2
Name: count, dtype: int64

=== MODEL SELECTION RESULTS ===
SelectionStatus
SELECTED              34
NO_VALIDATED_MODEL     1
Name: count, dtype: int64

Selected model distribution:
SelectedModel
ETS_Seasonal    28
MA_6             3
ARIMA            2
MA_3             1
Name: count, dtype: int64

Materials without validated model:


,MaterialKey,SelectedModel,MAE,RMSE,Bias,SelectionStatus
22,169,NaN,NaN,NaN,NaN,NO_VALIDATED_MODEL



=== SAMPLE SELECTED MODELS ===


,MaterialKey,SelectedModel,MAE,RMSE,Bias,SelectionStatus
0,4,ARIMA,863.600110,1207.014960,269.441546,SELECTED
1,5,MA_3,1422.760402,1778.940154,239.379515,SELECTED
2,6,ETS_Seasonal,258.128393,284.474018,106.598485,SELECTED
3,9,ETS_Seasonal,172.839914,195.183744,42.648064,SELECTED
4,11,ETS_Seasonal,1964.364296,2249.839821,1166.484297,SELECTED
5,12,ETS_Seasonal,982.081921,1124.744014,583.115181,SELECTED
6,13,ETS_Seasonal,691.335256,780.672761,170.596186,SELECTED
7,23,ETS_Seasonal,675.406758,763.429819,168.898374,SELECTED
8,24,ETS_Seasonal,675.406758,763.429819,168.898374,SELECTED
9,25,ETS_Seasonal,657.493070,743.220281,176.982601,SELECTED


Cell 5: INSPECT FINAL MODEL SELECTION

In [9]:
# ===== G11 - CELL 5: INSPECT FINAL MODEL SELECTION =====

print("=== FINAL MODEL SELECTION SUMMARY ===")

print("\nSelection status:")
print(
    model_selection["SelectionStatus"]
    .value_counts()
)

print("\nSelected model distribution:")
print(
    model_selection.loc[
        model_selection["SelectionStatus"] == "SELECTED",
        "SelectedModel"
    ]
    .value_counts()
)

print("\nTotal selected:")
print(
    (model_selection["SelectionStatus"] == "SELECTED").sum()
)

print("\nTotal without validated model:")
print(
    (model_selection["SelectionStatus"] == "NO_VALIDATED_MODEL").sum()
)

print("\n=== FULL SELECTED MODEL LIST ===")

display(
    model_selection[
        [
            "MaterialKey",
            "SelectedModel",
            "MAE",
            "RMSE",
            "Bias",
            "SelectionStatus"
        ]
    ]
)

=== FINAL MODEL SELECTION SUMMARY ===

Selection status:
SelectionStatus
SELECTED              34
NO_VALIDATED_MODEL     1
Name: count, dtype: int64

Selected model distribution:
SelectedModel
ETS_Seasonal    28
MA_6             3
ARIMA            2
MA_3             1
Name: count, dtype: int64

Total selected:
34

Total without validated model:
1

=== FULL SELECTED MODEL LIST ===


,MaterialKey,SelectedModel,MAE,RMSE,Bias,SelectionStatus
0,4,ARIMA,863.600110,1207.014960,269.441546,SELECTED
1,5,MA_3,1422.760402,1778.940154,239.379515,SELECTED
2,6,ETS_Seasonal,258.128393,284.474018,106.598485,SELECTED
3,9,ETS_Seasonal,172.839914,195.183744,42.648064,SELECTED
4,11,ETS_Seasonal,1964.364296,2249.839821,1166.484297,SELECTED
5,12,ETS_Seasonal,982.081921,1124.744014,583.115181,SELECTED
6,13,ETS_Seasonal,691.335256,780.672761,170.596186,SELECTED
7,23,ETS_Seasonal,675.406758,763.429819,168.898374,SELECTED
8,24,ETS_Seasonal,675.406758,763.429819,168.898374,SELECTED
9,25,ETS_Seasonal,657.493070,743.220281,176.982601,SELECTED


Cell 6: CHECK SELECTED MODELS AGAINST HISTORY COMPLETENESS

In [10]:
# ===== G11 - CELL 6: CHECK HISTORY COMPLETENESS =====

# -----------------------------------------
# Count observed months in full 35-month window
# -----------------------------------------

history_summary = (
    df.groupby(
        [
            "MaterialKey",
            "DSSEligibilityTier"
        ]
    )
    .agg(
        ObservedMonths=("Month", "count"),
        TrainObserved=("IsTrainPeriod", "sum"),
        TestObserved=("IsTestPeriod", "sum")
    )
    .reset_index()
)

history_summary["MissingMonths"] = (
    35 - history_summary["ObservedMonths"]
)

history_summary["HasComplete35Months"] = (
    history_summary["ObservedMonths"] == 35
)


# -----------------------------------------
# Attach selected model
# -----------------------------------------

selection_history = model_selection.merge(
    history_summary,
    on="MaterialKey",
    how="left"
)


print("=== SELECTED MODEL VS HISTORY COMPLETENESS ===")

print("\nTier distribution:")
print(
    selection_history["DSSEligibilityTier"]
    .value_counts()
)

print("\nComplete 35-month history:")
print(
    selection_history["HasComplete35Months"]
    .value_counts()
)

print("\n=== TIER B MATERIALS ===")

display(
    selection_history[
        selection_history["DSSEligibilityTier"] == "B"
    ][
        [
            "MaterialKey",
            "DSSEligibilityTier",
            "ObservedMonths",
            "MissingMonths",
            "TrainObserved",
            "TestObserved",
            "SelectedModel",
            "SelectionStatus"
        ]
    ]
)


print("\n=== SELECTED MODELS WITH INCOMPLETE HISTORY ===")

display(
    selection_history[
        (selection_history["SelectionStatus"] == "SELECTED")
        & (~selection_history["HasComplete35Months"])
    ][
        [
            "MaterialKey",
            "DSSEligibilityTier",
            "ObservedMonths",
            "MissingMonths",
            "SelectedModel"
        ]
    ]
)

=== SELECTED MODEL VS HISTORY COMPLETENESS ===

Tier distribution:
DSSEligibilityTier
A    28
B     7
Name: count, dtype: int64

Complete 35-month history:
HasComplete35Months
True     30
False     5
Name: count, dtype: int64

=== TIER B MATERIALS ===


,MaterialKey,DSSEligibilityTier,ObservedMonths,MissingMonths,TrainObserved,TestObserved,SelectedModel,SelectionStatus
0,4,B,12,23,6,6,ARIMA,SELECTED
2,6,B,35,0,29,6,ETS_Seasonal,SELECTED
20,167,B,24,11,18,6,MA_6,SELECTED
21,168,B,24,11,18,6,ARIMA,SELECTED
22,169,B,23,12,23,0,NaN,NO_VALIDATED_MODEL
24,171,B,35,0,29,6,ETS_Seasonal,SELECTED
26,173,B,12,23,6,6,MA_6,SELECTED



=== SELECTED MODELS WITH INCOMPLETE HISTORY ===


,MaterialKey,DSSEligibilityTier,ObservedMonths,MissingMonths,SelectedModel
0,4,B,12,23,ARIMA
20,167,B,24,11,MA_6
21,168,B,24,11,ARIMA
26,173,B,12,23,MA_6


Cell 7: CHECK CONTIGUOUS OBSERVED HISTORY

In [11]:
# ===== G11 - CELL 7: CHECK CONTIGUOUS OBSERVED HISTORY =====

def check_contiguous_months(material_df):
    """
    Check whether the observed months form one continuous
    monthly block, even if the full 35-month window is incomplete.
    """

    months = (
        material_df["Month"]
        .sort_values()
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if len(months) <= 1:
        return True

    expected = pd.date_range(
        start=months.iloc[0],
        end=months.iloc[-1],
        freq="MS"
    )

    return (
        len(months) == len(expected)
        and list(months) == list(expected)
    )


continuity_rows = []

for material_key, group in df.groupby("MaterialKey"):

    months = group["Month"].sort_values()

    continuity_rows.append({
        "MaterialKey": material_key,
        "FirstObservedMonth": months.min(),
        "LastObservedMonth": months.max(),
        "ObservedMonths": len(months),
        "IsContiguousObservedBlock":
            check_contiguous_months(group)
    })


continuity_check = pd.DataFrame(
    continuity_rows
)


selection_continuity = (
    selection_history.merge(
        continuity_check[
            [
                "MaterialKey",
                "FirstObservedMonth",
                "LastObservedMonth",
                "IsContiguousObservedBlock"
            ]
        ],
        on="MaterialKey",
        how="left"
    )
)


print("=== OBSERVED HISTORY CONTINUITY ===")

print(
    selection_continuity[
        "IsContiguousObservedBlock"
    ]
    .value_counts()
)


print("\n=== INCOMPLETE-HISTORY MATERIALS ===")

display(
    selection_continuity[
        ~selection_continuity["HasComplete35Months"]
    ][
        [
            "MaterialKey",
            "DSSEligibilityTier",
            "FirstObservedMonth",
            "LastObservedMonth",
            "ObservedMonths",
            "MissingMonths",
            "TrainObserved",
            "TestObserved",
            "IsContiguousObservedBlock",
            "SelectedModel",
            "SelectionStatus"
        ]
    ]
)

=== OBSERVED HISTORY CONTINUITY ===
IsContiguousObservedBlock
True    35
Name: count, dtype: int64

=== INCOMPLETE-HISTORY MATERIALS ===


,MaterialKey,DSSEligibilityTier,FirstObservedMonth,LastObservedMonth,ObservedMonths,MissingMonths,TrainObserved,TestObserved,IsContiguousObservedBlock,SelectedModel,SelectionStatus
0,4,B,2024-06-01,2025-05-01,12,23,6,6,True,ARIMA,SELECTED
20,167,B,2023-06-01,2025-05-01,24,11,18,6,True,MA_6,SELECTED
21,168,B,2023-06-01,2025-05-01,24,11,18,6,True,ARIMA,SELECTED
22,169,B,2022-07-01,2024-05-01,23,12,23,0,True,NaN,NO_VALIDATED_MODEL
26,173,B,2024-06-01,2025-05-01,12,23,6,6,True,MA_6,SELECTED


Cell 7B: ARIMA HELPER FOR FINAL REFIT

In [13]:
# ===== G11 - CELL 7B: ARIMA HELPER FOR FINAL REFIT =====

from itertools import product
from statsmodels.tsa.arima.model import ARIMA

# Same bounded search space used in G10
ARIMA_ORDERS = list(
    product(
        range(3),   # p = 0, 1, 2
        range(2),   # d = 0, 1
        range(3)    # q = 0, 1, 2
    )
)


def select_arima_order(train_values):
    """
    Select ARIMA(p,d,q) using the lowest AIC.

    This is the same bounded order-selection logic
    used in G10.
    """

    y = np.asarray(
        train_values,
        dtype=float
    )

    best_order = None
    best_aic = np.inf
    best_fit = None

    successful_candidates = 0
    failed_candidates = 0

    for order in ARIMA_ORDERS:

        try:

            model = ARIMA(
                y,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted = model.fit()

            if not np.isfinite(fitted.aic):
                failed_candidates += 1
                continue

            successful_candidates += 1

            if fitted.aic < best_aic:
                best_aic = fitted.aic
                best_order = order
                best_fit = fitted

        except Exception:
            failed_candidates += 1

    if best_fit is None:
        raise RuntimeError(
            "No ARIMA candidate could be fitted."
        )

    return (
        best_order,
        best_aic,
        best_fit,
        successful_candidates,
        failed_candidates
    )


print("=== ARIMA HELPER READY ===")
print(
    "Candidate orders:",
    len(ARIMA_ORDERS)
)

=== ARIMA HELPER READY ===
Candidate orders: 18


Cell 8: REFIT & GENERATE OFFICIAL 6-MONTH FORECAST

In [14]:
# ===== G11 - CELL 8: REFIT & GENERATE OFFICIAL FORECAST =====

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

OFFICIAL_HORIZON = 6

official_forecasts = []
official_logs = []


def recursive_ma_forecast(values, window, steps=6):
    """
    Recursive multi-step moving-average forecast.
    Same forecasting logic used in G06.
    """
    history = list(np.asarray(values, dtype=float))
    forecasts = []

    for _ in range(steps):
        forecast = np.mean(history[-window:])
        forecasts.append(forecast)
        history.append(forecast)

    return np.asarray(forecasts, dtype=float)


for _, selection in model_selection.iterrows():

    material_key = selection["MaterialKey"]
    selected_model = selection["SelectedModel"]
    selection_status = selection["SelectionStatus"]

    material_data = (
        df[df["MaterialKey"] == material_key]
        .sort_values("Month")
        .copy()
    )

    # -----------------------------------------
    # No validated model -> no official forecast
    # -----------------------------------------
    if selection_status != "SELECTED":

        official_logs.append({
            "MaterialKey": material_key,
            "SelectedModel": None,
            "ObservedMonths": len(material_data),
            "RefitStatus": "NO_VALIDATED_MODEL",
            "Message": (
                "Official forecast not generated "
                "because no model could be validated."
            )
        })

        continue

    y = (
        material_data["RequirementQty_PU"]
        .astype(float)
        .to_numpy()
    )

    last_month = material_data["Month"].max()

    future_months = pd.date_range(
        start=last_month + pd.offsets.MonthBegin(1),
        periods=OFFICIAL_HORIZON,
        freq="MS"
    )

    try:

        selected_order = None

        # -----------------------------------------
        # MA_3
        # -----------------------------------------
        if selected_model == "MA_3":

            raw_forecast = recursive_ma_forecast(
                y,
                window=3,
                steps=OFFICIAL_HORIZON
            )

        # -----------------------------------------
        # MA_6
        # -----------------------------------------
        elif selected_model == "MA_6":

            raw_forecast = recursive_ma_forecast(
                y,
                window=6,
                steps=OFFICIAL_HORIZON
            )

        # -----------------------------------------
        # ETS Seasonal
        # -----------------------------------------
        elif selected_model == "ETS_Seasonal":

            fitted = ExponentialSmoothing(
                y,
                trend="add",
                seasonal="add",
                seasonal_periods=12,
                initialization_method="estimated"
            ).fit()

            raw_forecast = np.asarray(
                fitted.forecast(OFFICIAL_HORIZON),
                dtype=float
            )

        # -----------------------------------------
        # ARIMA
        # Re-select order using full observed history
        # and TRAIN-INDEPENDENT AIC logic
        # -----------------------------------------
        elif selected_model == "ARIMA":

            (
                selected_order,
                best_aic,
                fitted,
                successful_candidates,
                failed_candidates
            ) = select_arima_order(y)

            raw_forecast = np.asarray(
                fitted.forecast(steps=OFFICIAL_HORIZON),
                dtype=float
            )

        else:
            raise ValueError(
                f"Unsupported selected model: {selected_model}"
            )

        # -----------------------------------------
        # Business constraint
        # -----------------------------------------
        final_forecast = np.maximum(
            raw_forecast,
            0
        )

        negative_count = int(
            (raw_forecast < 0).sum()
        )

        # -----------------------------------------
        # Save official forecast
        # -----------------------------------------
        for month, raw_f, forecast in zip(
            future_months,
            raw_forecast,
            final_forecast
        ):

            official_forecasts.append({
                "MaterialKey": material_key,
                "ForecastMonth": month,
                "SelectedModel": selected_model,
                "SelectedOrder": (
                    str(selected_order)
                    if selected_order is not None
                    else None
                ),
                "RawForecastQty": raw_f,
                "ForecastQty": forecast,
                "WasClipped": int(raw_f < 0)
            })

        official_logs.append({
            "MaterialKey": material_key,
            "SelectedModel": selected_model,
            "ObservedMonths": len(material_data),
            "RefitStatus": "SUCCESS",
            "Message": (
                f"{negative_count} negative forecast(s) clipped to 0"
                if negative_count > 0
                else ""
            )
        })

    except Exception as e:

        official_logs.append({
            "MaterialKey": material_key,
            "SelectedModel": selected_model,
            "ObservedMonths": len(material_data),
            "RefitStatus": "REFIT_FAILED",
            "Message": f"{type(e).__name__}: {str(e)}"
        })


official_forecasts = pd.DataFrame(
    official_forecasts
)

official_logs = pd.DataFrame(
    official_logs
)


print("=== OFFICIAL FORECAST GENERATION ===")

print(
    "Forecast rows:",
    len(official_forecasts)
)

print(
    "Materials forecasted:",
    official_forecasts["MaterialKey"].nunique()
)

print(
    "Forecast period:",
    official_forecasts["ForecastMonth"].min(),
    "to",
    official_forecasts["ForecastMonth"].max()
)

print("\nRefit status:")
print(
    official_logs["RefitStatus"]
    .value_counts()
)

print(
    "\nNegative forecasts clipped:",
    official_forecasts["WasClipped"].sum()
)

print("\nMaterials not successfully forecasted:")

display(
    official_logs[
        official_logs["RefitStatus"] != "SUCCESS"
    ]
)

d:\02_TTDN\FINAL\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\02_TTDN\FINAL\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\02_TTDN\FINAL\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\02_TTDN\FINAL\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\02_TTDN\FINAL\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\02

=== OFFICIAL FORECAST GENERATION ===
Forecast rows: 204
Materials forecasted: 34
Forecast period: 2025-06-01 00:00:00 to 2025-11-01 00:00:00

Refit status:
RefitStatus
SUCCESS               34
NO_VALIDATED_MODEL     1
Name: count, dtype: int64

Negative forecasts clipped: 0

Materials not successfully forecasted:


,MaterialKey,SelectedModel,ObservedMonths,RefitStatus,Message
22,169,NaN,23,NO_VALIDATED_MODEL,Official forecast not generated because no mod...


Cell 9: OFFICIAL FORECAST QA

In [15]:
# ===== G11 - CELL 9: OFFICIAL FORECAST QA =====

print("=== OFFICIAL FORECAST QA ===")

# -----------------------------------------
# Basic QA
# -----------------------------------------

print(
    "Forecast rows:",
    len(official_forecasts)
)

print(
    "Materials forecasted:",
    official_forecasts["MaterialKey"].nunique()
)

print(
    "Duplicate Material-Month:",
    official_forecasts.duplicated(
        subset=[
            "MaterialKey",
            "ForecastMonth"
        ]
    ).sum()
)

print(
    "NULL ForecastQty:",
    official_forecasts["ForecastQty"]
    .isna()
    .sum()
)

print(
    "Negative ForecastQty:",
    (
        official_forecasts["ForecastQty"] < 0
    ).sum()
)


# -----------------------------------------
# Forecast horizon QA
# -----------------------------------------

forecast_count = (
    official_forecasts
    .groupby("MaterialKey")
    .size()
)

print(
    "Materials with exactly 6 forecast months:",
    (forecast_count == 6).sum()
)

print(
    "Materials without exactly 6 forecast months:",
    (forecast_count != 6).sum()
)


# -----------------------------------------
# Month coverage
# -----------------------------------------

print("\nForecast months:")

print(
    official_forecasts["ForecastMonth"]
    .value_counts()
    .sort_index()
)


# -----------------------------------------
# Model distribution in official forecast
# -----------------------------------------

print("\nOfficial model distribution:")

print(
    official_forecasts[
        [
            "MaterialKey",
            "SelectedModel"
        ]
    ]
    .drop_duplicates()
    ["SelectedModel"]
    .value_counts()
)


# -----------------------------------------
# ARIMA final orders
# -----------------------------------------

print("\nFinal ARIMA orders:")

display(
    official_forecasts[
        official_forecasts["SelectedModel"]
        == "ARIMA"
    ][
        [
            "MaterialKey",
            "SelectedModel",
            "SelectedOrder"
        ]
    ]
    .drop_duplicates()
)


# -----------------------------------------
# Material 169 policy check
# -----------------------------------------

print("\nMaterial 169 forecast rows:")

print(
    len(
        official_forecasts[
            official_forecasts["MaterialKey"] == 169
        ]
    )
)


# -----------------------------------------
# Sample official forecasts
# -----------------------------------------

print("\n=== SAMPLE OFFICIAL FORECAST ===")

display(
    official_forecasts.head(12)
)

=== OFFICIAL FORECAST QA ===
Forecast rows: 204
Materials forecasted: 34
Duplicate Material-Month: 0
NULL ForecastQty: 0
Negative ForecastQty: 0
Materials with exactly 6 forecast months: 34
Materials without exactly 6 forecast months: 0

Forecast months:
ForecastMonth
2025-06-01    34
2025-07-01    34
2025-08-01    34
2025-09-01    34
2025-10-01    34
2025-11-01    34
Name: count, dtype: int64

Official model distribution:
SelectedModel
ETS_Seasonal    28
MA_6             3
ARIMA            2
MA_3             1
Name: count, dtype: int64

Final ARIMA orders:


,MaterialKey,SelectedModel,SelectedOrder
0,4,ARIMA,"(0, 1, 2)"
126,168,ARIMA,"(0, 1, 2)"



Material 169 forecast rows:
0

=== SAMPLE OFFICIAL FORECAST ===


,MaterialKey,ForecastMonth,SelectedModel,SelectedOrder,RawForecastQty,ForecastQty,WasClipped
0,4,2025-06-01,ARIMA,"(0, 1, 2)",1022.287637,1022.287637,0
1,4,2025-07-01,ARIMA,"(0, 1, 2)",2156.249332,2156.249332,0
2,4,2025-08-01,ARIMA,"(0, 1, 2)",2156.249332,2156.249332,0
3,4,2025-09-01,ARIMA,"(0, 1, 2)",2156.249332,2156.249332,0
4,4,2025-10-01,ARIMA,"(0, 1, 2)",2156.249332,2156.249332,0
5,4,2025-11-01,ARIMA,"(0, 1, 2)",2156.249332,2156.249332,0
6,5,2025-06-01,MA_3,NaN,4201.333333,4201.333333,0
7,5,2025-07-01,MA_3,NaN,4475.111111,4475.111111,0
8,5,2025-08-01,MA_3,NaN,4390.814815,4390.814815,0
9,5,2025-09-01,MA_3,NaN,4355.753086,4355.753086,0


Cell 10: EXPORT FINAL FORECAST OUTPUTS

In [16]:
# ===== G11 - CELL 10: EXPORT FINAL FORECAST OUTPUTS =====

# -----------------------------------------
# Sort final outputs
# -----------------------------------------

model_selection = (
    model_selection
    .sort_values("MaterialKey")
    .reset_index(drop=True)
)

official_forecasts = (
    official_forecasts
    .sort_values(
        ["MaterialKey", "ForecastMonth"]
    )
    .reset_index(drop=True)
)

official_logs = (
    official_logs
    .sort_values("MaterialKey")
    .reset_index(drop=True)
)


# -----------------------------------------
# Export
# -----------------------------------------

model_selection.to_csv(
    "../output/G11_model_selection.csv",
    index=False
)

official_forecasts.to_csv(
    "../output/G11_official_forecast.csv",
    index=False
)

official_logs.to_csv(
    "../output/G11_official_forecast_logs.csv",
    index=False
)


# -----------------------------------------
# Final export QA
# -----------------------------------------

print("=== G11 EXPORT COMPLETED ===")

print(
    "Model selection rows:",
    len(model_selection)
)

print(
    "Official forecast rows:",
    len(official_forecasts)
)

print(
    "Official forecast materials:",
    official_forecasts["MaterialKey"].nunique()
)

print(
    "Official log rows:",
    len(official_logs)
)

print(
    "Selected models:",
    (
        model_selection["SelectionStatus"]
        == "SELECTED"
    ).sum()
)

print(
    "No validated model:",
    (
        model_selection["SelectionStatus"]
        == "NO_VALIDATED_MODEL"
    ).sum()
)

print(
    "Successful refits:",
    (
        official_logs["RefitStatus"]
        == "SUCCESS"
    ).sum()
)

print(
    "Refit failures:",
    (
        official_logs["RefitStatus"]
        == "REFIT_FAILED"
    ).sum()
)

print(
    "Duplicate official forecasts:",
    official_forecasts.duplicated(
        subset=[
            "MaterialKey",
            "ForecastMonth"
        ]
    ).sum()
)

=== G11 EXPORT COMPLETED ===
Model selection rows: 35
Official forecast rows: 204
Official forecast materials: 34
Official log rows: 35
Selected models: 34
No validated model: 1
Successful refits: 34
Refit failures: 0
Duplicate official forecasts: 0


Cell 11: PREPARE DW LOAD FILES

In [19]:
# ===== G11 - CELL 11: PREPARE DW LOAD FILES =====

# -----------------------------------------
# 1. Attach material / purchase UoM info
# -----------------------------------------

material_info = (
    df[
        [
            "MaterialKey",
            "ProductNumber",
            "MaterialName",
            "UnitCode"
        ]
    ]
    .drop_duplicates(subset=["MaterialKey"])
)

dw_forecast = official_forecasts.merge(
    material_info,
    on="MaterialKey",
    how="left"
)


# -----------------------------------------
# 2. Map purchase unit -> UoMKey
# Confirmed from Dim_UnitMeasure:
# DZ = 16
# EA = 17
# -----------------------------------------

UOM_KEY_MAP = {
    "DZ": 16,
    "EA": 17
}

dw_forecast["UoMKey"] = (
    dw_forecast["UnitCode"]
    .map(UOM_KEY_MAP)
)


# -----------------------------------------
# 3. Map monthly date -> DateKey
# Monthly grain uses first day of month
# -----------------------------------------

dw_forecast["ForecastMonthKey"] = (
    dw_forecast["ForecastMonth"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)


# -----------------------------------------
# 4. Prepare final DW staging structure
# ForecastRunKey will be assigned in SQL
# after Dim_ForecastRun is inserted.
# -----------------------------------------

dw_forecast_load = dw_forecast[
    [
        "MaterialKey",
        "ForecastMonthKey",
        "SelectedModel",
        "UoMKey",
        "ForecastQty"
    ]
].copy()


# ForecastQty is already in purchase unit
dw_forecast_load["ForecastQtyPurchaseUoM"] = (
    dw_forecast_load["ForecastQty"]
)


# -----------------------------------------
# 5. QA
# -----------------------------------------

print("=== DW FORECAST LOAD PREPARATION ===")

print(
    "Rows:",
    len(dw_forecast_load)
)

print(
    "Materials:",
    dw_forecast_load["MaterialKey"].nunique()
)

print(
    "Months:",
    dw_forecast_load["ForecastMonthKey"].nunique()
)

print(
    "NULL UoMKey:",
    dw_forecast_load["UoMKey"].isna().sum()
)

print(
    "NULL ForecastQty:",
    dw_forecast_load["ForecastQty"].isna().sum()
)

print(
    "Duplicate staging key:",
    dw_forecast_load.duplicated(
        subset=[
            "MaterialKey",
            "ForecastMonthKey",
            "SelectedModel",
            "UoMKey"
        ]
    ).sum()
)

print("\nModel distribution:")

print(
    dw_forecast_load[
        [
            "MaterialKey",
            "SelectedModel"
        ]
    ]
    .drop_duplicates()
    ["SelectedModel"]
    .value_counts()
)

# ===== CONTINUE CELL 11 QA =====

print("\nUoM distribution:")

print(
    dw_forecast_load[
        ["MaterialKey", "UoMKey"]
    ]
    .drop_duplicates()
    ["UoMKey"]
    .value_counts()
    .sort_index()
)

print("\nForecast month keys:")

print(
    dw_forecast_load["ForecastMonthKey"]
    .value_counts()
    .sort_index()
)


# -----------------------------------------
# Export SQL staging file
# -----------------------------------------

dw_forecast_load.to_csv(
    "../output/G11_DW_demand_forecast_load.csv",
    index=False
)

print(
    "\nExported: "
    "../output/G11_DW_demand_forecast_load.csv"
)

=== DW FORECAST LOAD PREPARATION ===
Rows: 204
Materials: 34
Months: 6
NULL UoMKey: 0
NULL ForecastQty: 0
Duplicate staging key: 0

Model distribution:
SelectedModel
ETS_Seasonal    28
MA_6             3
ARIMA            2
MA_3             1
Name: count, dtype: int64

UoM distribution:
UoMKey
16     4
17    30
Name: count, dtype: int64

Forecast month keys:
ForecastMonthKey
20250601    34
20250701    34
20250801    34
20250901    34
20251001    34
20251101    34
Name: count, dtype: int64

Exported: ../output/G11_DW_demand_forecast_load.csv


Cell 12: Prepare Evaluation DW Load

In [22]:
# ===== G11 - CELL 12: PREPARE DW FORECAST EVALUATION =====

import pandas as pd

# =========================================================
# 1. LOAD TEST PREDICTIONS DIRECTLY FROM G06 / G09 / G10
# =========================================================

g06_pred_dw = pd.read_csv(
    "../output/G06_baseline_predictions.csv"
)

g09_pred_dw = pd.read_csv(
    "../output/G09_ets_predictions.csv"
)

g10_pred_dw = pd.read_csv(
    "../output/G10_arima_predictions.csv"
)

print("=== SOURCE PREDICTION FILES ===")
print("G06 rows:", len(g06_pred_dw))
print("G09 rows:", len(g09_pred_dw))
print("G10 rows:", len(g10_pred_dw))

print("\nG06 columns:")
print(g06_pred_dw.columns.tolist())

print("\nG09 columns:")
print(g09_pred_dw.columns.tolist())

print("\nG10 columns:")
print(g10_pred_dw.columns.tolist())


# =========================================================
# 2. STANDARDIZE COLUMN NAMES
# =========================================================

def standardize_prediction_columns(pred_df):
    """
    Standardize prediction output columns from G06/G09/G10
    into:
        MaterialKey
        Month
        Model
        Actual
        Forecast
    """

    x = pred_df.copy()

    # ----- Month column -----
    month_candidates = [
        "Month",
        "TestMonth",
        "MonthStartDate",
        "ForecastMonth"
    ]

    month_col = next(
        (c for c in month_candidates if c in x.columns),
        None
    )

    if month_col is None:
        raise ValueError(
            f"Cannot find month column. "
            f"Available columns: {x.columns.tolist()}"
        )

    if month_col != "Month":
        x = x.rename(columns={month_col: "Month"})


    # ----- Model column -----
    model_candidates = [
        "Model",
        "ModelName"
    ]

    model_col = next(
        (c for c in model_candidates if c in x.columns),
        None
    )

    if model_col is None:
        raise ValueError(
            f"Cannot find model column. "
            f"Available columns: {x.columns.tolist()}"
        )

    if model_col != "Model":
        x = x.rename(columns={model_col: "Model"})


    # ----- Actual column -----
    actual_candidates = [
        "Actual",
        "ActualQty",
        "RequirementQty_PU"
    ]

    actual_col = next(
        (c for c in actual_candidates if c in x.columns),
        None
    )

    if actual_col is None:
        raise ValueError(
            f"Cannot find actual column. "
            f"Available columns: {x.columns.tolist()}"
        )

    if actual_col != "Actual":
        x = x.rename(columns={actual_col: "Actual"})


    # ----- Forecast column -----
    forecast_candidates = [
        "Forecast",
        "ForecastQty",
        "FinalForecast",
        "ForecastFinal"
    ]

    forecast_col = next(
        (c for c in forecast_candidates if c in x.columns),
        None
    )

    if forecast_col is None:
        raise ValueError(
            f"Cannot find forecast column. "
            f"Available columns: {x.columns.tolist()}"
        )

    if forecast_col != "Forecast":
        x = x.rename(columns={forecast_col: "Forecast"})


    # ----- Required columns -----
    required = [
        "MaterialKey",
        "Month",
        "Model",
        "Actual",
        "Forecast"
    ]

    missing = [
        c for c in required
        if c not in x.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    return x[required].copy()


g06_std = standardize_prediction_columns(g06_pred_dw)
g09_std = standardize_prediction_columns(g09_pred_dw)
g10_std = standardize_prediction_columns(g10_pred_dw)


# =========================================================
# 3. COMBINE ALL BACKTEST PREDICTIONS
# =========================================================

all_predictions = pd.concat(
    [
        g06_std,
        g09_std,
        g10_std
    ],
    ignore_index=True
)

all_predictions["Month"] = pd.to_datetime(
    all_predictions["Month"]
)

print("\n=== COMBINED TEST PREDICTIONS ===")
print("Rows:", len(all_predictions))
print(
    "Materials:",
    all_predictions["MaterialKey"].nunique()
)

print(
    "Models:",
    sorted(all_predictions["Model"].unique())
)


# =========================================================
# 4. GET G11 SELECTED MODEL FOR EACH MATERIAL
# =========================================================

selected_models = model_selection.loc[
    model_selection["SelectionStatus"] == "SELECTED",
    [
        "MaterialKey",
        "SelectedModel"
    ]
].copy()

print("\n=== SELECTED MODELS ===")
print(
    "Selected materials:",
    selected_models["MaterialKey"].nunique()
)

print(
    selected_models["SelectedModel"]
    .value_counts()
)


# =========================================================
# 5. KEEP ONLY BACKTEST PREDICTION OF SELECTED MODEL
# =========================================================

selected_eval = all_predictions.merge(
    selected_models,
    left_on=[
        "MaterialKey",
        "Model"
    ],
    right_on=[
        "MaterialKey",
        "SelectedModel"
    ],
    how="inner"
)

print("\nSelected evaluation rows:", len(selected_eval))


# =========================================================
# 6. CREATE TEST MONTH KEY
#
# Dim_Date is daily grain.
# Monthly forecast/evaluation uses first day of month:
# 2024-12 -> 20241201
# ...
# 2025-05 -> 20250501
# =========================================================

selected_eval["TestMonthKey"] = (
    selected_eval["Month"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)


# =========================================================
# 7. PREPARE FACT_FORECASTEVALUATION STAGING DATA
#
# ForecastRunKey is NOT assigned here.
# SQL will assign it after inserting Dim_ForecastRun.
# =========================================================

dw_evaluation_load = selected_eval[
    [
        "MaterialKey",
        "TestMonthKey",
        "SelectedModel",
        "Actual",
        "Forecast"
    ]
].copy()

dw_evaluation_load = dw_evaluation_load.rename(
    columns={
        "Actual": "ActualQty",
        "Forecast": "ForecastQty"
    }
)


# =========================================================
# 8. DATA TYPE CLEANUP
# =========================================================

dw_evaluation_load["MaterialKey"] = (
    dw_evaluation_load["MaterialKey"]
    .astype(int)
)

dw_evaluation_load["TestMonthKey"] = (
    dw_evaluation_load["TestMonthKey"]
    .astype(int)
)

dw_evaluation_load["ActualQty"] = pd.to_numeric(
    dw_evaluation_load["ActualQty"],
    errors="coerce"
)

dw_evaluation_load["ForecastQty"] = pd.to_numeric(
    dw_evaluation_load["ForecastQty"],
    errors="coerce"
)


# =========================================================
# 9. QA CHECKS
# =========================================================

print("\n=== DW FORECAST EVALUATION PREPARATION ===")

print(
    "Rows:",
    len(dw_evaluation_load)
)

print(
    "Materials:",
    dw_evaluation_load["MaterialKey"].nunique()
)

print(
    "Months:",
    dw_evaluation_load["TestMonthKey"].nunique()
)

print(
    "NULL ActualQty:",
    dw_evaluation_load["ActualQty"].isna().sum()
)

print(
    "NULL ForecastQty:",
    dw_evaluation_load["ForecastQty"].isna().sum()
)

duplicate_count = (
    dw_evaluation_load
    .duplicated(
        subset=[
            "MaterialKey",
            "TestMonthKey",
            "SelectedModel"
        ]
    )
    .sum()
)

print(
    "Duplicate staging key:",
    duplicate_count
)


# =========================================================
# 10. MODEL DISTRIBUTION
# =========================================================

print("\nModel distribution:")

model_distribution = (
    dw_evaluation_load[
        [
            "MaterialKey",
            "SelectedModel"
        ]
    ]
    .drop_duplicates()
    ["SelectedModel"]
    .value_counts()
)

print(model_distribution)


# =========================================================
# 11. TEST MONTH DISTRIBUTION
# =========================================================

print("\nTest month distribution:")

month_distribution = (
    dw_evaluation_load["TestMonthKey"]
    .value_counts()
    .sort_index()
)

print(month_distribution)


# =========================================================
# 12. MATERIAL 169 CHECK
# Must NOT be loaded because it has no valid 6-month test
# =========================================================

material_169_rows = (
    dw_evaluation_load[
        dw_evaluation_load["MaterialKey"] == 169
    ]
)

print(
    "\nMaterial 169 evaluation rows:",
    len(material_169_rows)
)


# =========================================================
# 13. FINAL PASS / FAIL
# =========================================================

expected_months = {
    20241201,
    20250101,
    20250201,
    20250301,
    20250401,
    20250501
}

qa_pass = (
    len(dw_evaluation_load) == 204
    and dw_evaluation_load["MaterialKey"].nunique() == 34
    and dw_evaluation_load["TestMonthKey"].nunique() == 6
    and dw_evaluation_load["ActualQty"].isna().sum() == 0
    and dw_evaluation_load["ForecastQty"].isna().sum() == 0
    and duplicate_count == 0
    and set(
        dw_evaluation_load["TestMonthKey"].unique()
    ) == expected_months
    and len(material_169_rows) == 0
)

print("\n=== FINAL QA ===")

if qa_pass:
    print("PASS - DW forecast evaluation staging is valid.")
else:
    print("FAIL - Check QA results above before export.")


# =========================================================
# 14. EXPORT ONLY IF QA PASSES
# =========================================================

if qa_pass:

    output_path = (
        "../output/"
        "G11_DW_forecast_evaluation_load.csv"
    )

    dw_evaluation_load.to_csv(
        output_path,
        index=False
    )

    print(
        "\nExported:",
        output_path
    )

else:

    print(
        "\nNOT EXPORTED because QA did not pass."
    )

=== SOURCE PREDICTION FILES ===
G06 rows: 612
G09 rows: 588
G10 rows: 204

G06 columns:
['MaterialKey', 'ForecastMonth', 'Model', 'ActualQty', 'ForecastQty', 'Error']

G09 columns:
['MaterialKey', 'ForecastMonth', 'Model', 'ActualQty', 'RawForecastQty', 'ForecastQty', 'WasClipped', 'Error']

G10 columns:
['MaterialKey', 'ForecastMonth', 'Model', 'Order', 'ActualQty', 'RawForecastQty', 'ForecastQty', 'WasClipped', 'Error']

=== COMBINED TEST PREDICTIONS ===
Rows: 1404
Materials: 34
Models: ['ARIMA', 'ETS_Level', 'ETS_Seasonal', 'ETS_Trend', 'MA_3', 'MA_6', 'Naive']

=== SELECTED MODELS ===
Selected materials: 34
SelectedModel
ETS_Seasonal    28
MA_6             3
ARIMA            2
MA_3             1
Name: count, dtype: int64

Selected evaluation rows: 204

=== DW FORECAST EVALUATION PREPARATION ===
Rows: 204
Materials: 34
Months: 6
NULL ActualQty: 0
NULL ForecastQty: 0
Duplicate staging key: 0

Model distribution:
SelectedModel
ETS_Seasonal    28
MA_6             3
ARIMA            2
M